In [3]:
!pip install -q imagehash opencv-python-headless tqdm kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 13.8 MB/s eta 0:00:00


In [11]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.5 MB/s eta 0:00:00


In [4]:
# ── Environment Setup ──────────────────────────────────────────────────────
import os
import json
import logging
from pathlib import Path
import sys

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', stream=sys.stdout, force=True)

os.environ['KAGGLE_USERNAME'] = "paranietharan29"
os.environ['KAGGLE_KEY']      = "KGAT_21968fa3538f832e53f6adc53b096497"

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DATA_PATH  = Path("/content/raw_data")
DATASET_PATH   = Path("/content/dataset")
QUARANTINE_PATH = Path("/content/quarantine")

for p in [RAW_DATA_PATH, DATASET_PATH, QUARANTINE_PATH]:
    p.mkdir(parents=True, exist_ok=True)

logging.info("Environment setup complete.")

INFO: Environment setup complete.


In [5]:
# ── Imports ────────────────────────────────────────────────────────────────
import zipfile
import shutil
import numpy as np
import imagehash
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

INFO: NumExpr defaulting to 2 threads.


In [ ]:
import torch
from pathlib import Path

# TRAINING CONFIGURATION
MODEL_TYPE = 'yolov8n.pt' # 'yolov8n.pt' for speed, 'yolov8s.pt' for better accuracy
EPOCHS = 50
PATIENCE = 10 # Early stopping patience
IMG_SIZE = 640
BATCH_SIZE = 16

In [6]:
# ── Download Dataset ───────────────────────────────────────────────────────
def download_dataset():
    logging.info("entering into download dataset")
    zip_path = RAW_DATA_PATH / "rdd-2022.zip"
    existing_folders = [p for p in RAW_DATA_PATH.iterdir() if p.is_dir()]
    if existing_folders:
        logging.info(f"Dataset already extracted ({len(existing_folders)} folders found)")
        # delete the fodler
        for folder in existing_folders:
            logging.info(f"Deleting folder: {folder}")
            shutil.rmtree(folder)

    logging.info("Downloading RDD-2022 dataset (~9.9 GB)...")
    os.system(f"kaggle datasets download -d aliabdelmenam/rdd-2022 -p {RAW_DATA_PATH}")

    if zip_path.exists():
        logging.info("Extracting zip...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(RAW_DATA_PATH)
        zip_path.unlink()  # delete zip after extraction to free space
        logging.info("Extraction complete. Zip deleted to save space.")
    else:
        logging.error("Download failed — zip not found. Check your Kaggle credentials.")

In [7]:
# ── VOC → YOLO Conversion ──────────────────────────────────────────────────
# RDD-2022 damage classes: D00=longitudinal crack, D10=transverse crack,
# D20=alligator crack, D40=pothole. All mapped to class 0 ('damage').
TARGET_CLASS_MAP = {"D00": 0, "D10": 0, "D20": 0, "D40": 0}

def convert_voc_to_yolo(xml_file, target_class_map=None):
    """Parse a PascalVOC XML and return YOLO-format label strings.

    FIX: uses explicit named variables for coordinates instead of a
    positional tuple (b[0]..b[3]) which was error-prone and confusing.
    The math is identical but now self-documenting and safe.
    """
    if target_class_map is None:
        target_class_map = TARGET_CLASS_MAP

    try:
        tree = ET.parse(xml_file)
    except ET.ParseError as e:
        logging.warning(f"Malformed XML skipped: {xml_file} — {e}")
        return []

    root = tree.getroot()
    size = root.find('size')
    if size is None:
        return []

    w = int(size.find('width').text)
    h = int(size.find('height').text)
    if w == 0 or h == 0:
        return []

    yolo_labels = []
    for obj in root.iter('object'):
        cls_name = obj.find('name').text
        if cls_name not in target_class_map:
            continue

        cls_id = target_class_map[cls_name]
        xmlbox = obj.find('bndbox')

        # FIX: explicit names — no more confusing b[0]..b[3] indexing
        xmin = float(xmlbox.find('xmin').text)
        xmax = float(xmlbox.find('xmax').text)
        ymin = float(xmlbox.find('ymin').text)
        ymax = float(xmlbox.find('ymax').text)

        # YOLO format: x_center y_center width height (all normalised 0–1)
        x_center = (xmin + xmax) / (2.0 * w)
        y_center = (ymin + ymax) / (2.0 * h)
        bbox_w   = (xmax - xmin) / w
        bbox_h   = (ymax - ymin) / h

        # Clamp to [0, 1] to guard against annotation errors
        x_center = max(0.0, min(1.0, x_center))
        y_center = max(0.0, min(1.0, y_center))
        bbox_w   = max(0.0, min(1.0, bbox_w))
        bbox_h   = max(0.0, min(1.0, bbox_h))

        yolo_labels.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {bbox_w:.6f} {bbox_h:.6f}")

    return yolo_labels

In [8]:
def clean_and_process():
    """
    The Kaggle aliabdelmenam/rdd-2022 dataset is already in YOLO format.
    Labels are .txt files in train/labels/ alongside train/images/.
    Just validate image integrity and pair each image with its label.
    """
    split_root = RAW_DATA_PATH / "RDD_SPLIT"
    image_files = list((split_root / "train" / "images").glob("*.jpg"))
    print(f"Found {len(image_files)} images")

    if not image_files:
        logging.error("No images found in RDD_SPLIT/train/images/")
        return []

    valid_samples = []
    stats = {'integrity_fail': 0, 'no_label': 0, 'empty_label': 0}

    for img_path in tqdm(image_files):
        # Integrity check
        try:
            with Image.open(img_path) as im:
                im.verify()
        except Exception:
            stats['integrity_fail'] += 1
            continue

        # Find paired .txt label file
        label_path = split_root / "train" / "labels" / img_path.with_suffix(".txt").name
        if not label_path.exists():
            stats['no_label'] += 1
            continue

        # Read existing YOLO labels — filter to class 0 only (damage)
        lines = [l.strip() for l in label_path.read_text().splitlines() if l.strip()]
        labels = [l for l in lines if l.startswith("0 ")]  # keep class 0 only

        if not labels:
            stats['empty_label'] += 1
            continue

        valid_samples.append({'img': img_path, 'labels': labels})

    logging.info(f"Done. Valid: {len(valid_samples)}, Stats: {stats}")
    return valid_samples

In [9]:
# ── Dataset Splitting & YOLO Config ───────────────────────────────────────
def finalize_dataset(samples):
    if not samples:
        logging.error("Cannot finalize dataset: samples list is empty!")
        return

    train, temp = train_test_split(samples, test_size=0.30, random_state=42)
    val,   test = train_test_split(temp,    test_size=0.33, random_state=42)

    logging.info(f"Split sizes — train: {len(train)}, val: {len(val)}, test: {len(test)}")

    splits = {'train': train, 'val': val, 'test': test}
    for split_name, data in splits.items():
        img_dir   = DATASET_PATH / split_name / 'images'
        label_dir = DATASET_PATH / split_name / 'labels'
        img_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)

        for item in data:
            shutil.copy(item['img'], img_dir / item['img'].name)
            label_path = label_dir / f"{item['img'].stem}.txt"
            with open(label_path, 'w') as f:
                f.write("\n".join(item['labels']))

    # data.yaml — using absolute paths (required for Colab/YOLO)
    yaml_content = (
        f"path: {DATASET_PATH}\n"
        f"train: train/images\n"
        f"val:   val/images\n"
        f"test:  test/images\n"
        f"nc: 1\n"
        f"names: ['road_damage']\n"
    )
    yaml_path = DATASET_PATH / 'data.yaml'
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)

    logging.info(f"Dataset finalised. YAML written to {yaml_path}")

In [10]:
download_dataset()

INFO: entering into download dataset
INFO: Downloading RDD-2022 dataset (~9.9 GB)...
INFO: Extracting zip...
INFO: Extraction complete. Zip deleted to save space.


In [13]:
valid_data = clean_and_process()

Found 26869 images


100%|██████████| 26869/26869 [00:47<00:00, 564.69it/s]

INFO: Done. Valid: 9457, Stats: {'integrity_fail': 0, 'no_label': 0, 'empty_label': 17412}


In [15]:
# ── Finalize Dataset
finalize_dataset(valid_data)

INFO: Split sizes — train: 6619, val: 1901, test: 937
INFO: Dataset finalised. YAML written to /content/dataset/data.yaml


In [17]:
import torch
from ultralytics import YOLO
from pathlib import Path

# ── Train YOLO Model (Restoring device and hyperparameters) ────────────────
# Using configuration from cell 10ba6e98

# Initialize model
model = YOLO(MODEL_TYPE)

# Start training with full parameterization
results = model.train(
    data=str(DATASET_PATH / "data.yaml"),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device="0" if torch.cuda.is_available() else "cpu",
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    augment=True,
    cache=False,
    workers=2,
    project="/content/runs",
    name="rdd2022_yolov8_optimized",
    exist_ok=True
)

# Capture best weights path
best_weights = Path(results.save_dir) / 'weights' / 'best.pt'
print(f"\n✅ Training complete. Best weights: {best_weights}")

In [ ]:
# ── Validate on val set ───────────────────────────────────────────────────
metrics = model.val()
print("mAP50:    ", metrics.box.map50)
print("mAP50-95: ", metrics.box.map)

In [12]:
from google.colab import drive
import shutil
from pathlib import Path

def save_model_to_drive(model_source_path, drive_folder_name="RDD2022_YOLO_Models"):
    logging.info("Mounting Google Drive...")
    drive.mount('/content/drive')

    drive_dest_dir = Path(f"/content/drive/MyDrive/{drive_folder_name}")
    drive_dest_dir.mkdir(parents=True, exist_ok=True)

    model_filename = Path(model_source_path).name
    dest_path = drive_dest_dir / model_filename

    if Path(model_source_path).exists():
        logging.info(f"Copying model from {model_source_path} to {dest_path}")
        shutil.copy(model_source_path, dest_path)
        logging.info(f"Model saved to Google Drive: {dest_path}")
    else:
        logging.error(f"Model file not found at {model_source_path}. Please ensure training completed successfully and the path is correct.")


In [ ]:
save_model_to_drive(best_weights)

In [16]:
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path
import glob
import logging

logging.info("Mounting Google Drive to load the model...")
drive.mount('/content/drive')

# Define the path to the model saved on Google Drive
# This path should match where save_model_to_drive saved the 'best.pt' file.
drive_model_path = Path('/content/drive/MyDrive/RDD2022_YOLO_Models/best.pt')

if drive_model_path.exists():
    logging.info(f"Loading model from Google Drive: {drive_model_path}")
    loaded_model = YOLO(drive_model_path)

    # Get a few test images from the previously prepared DATASET_PATH
    test_images = glob.glob(str(DATASET_PATH / "test/images/*.jpg"))
    if not test_images:
        logging.error("No test images found in the dataset's test split. Please ensure the dataset was finalized correctly.")
    else:
        # Take up to 5 test images for demonstration
        sample_test_images = test_images[:5]
        logging.info(f"Running inference on {len(sample_test_images)} sample test images...")
        # Use the loaded_model to make predictions
        # Setting save=True will save annotated images to /content/runs/predict_from_drive/
        results_from_drive = loaded_model.predict(sample_test_images, conf=0.25, save=True, project="/content/runs", name="predict_from_drive")
        print(f"\nPredictions saved to /content/runs/predict_from_drive/ for the following images:")
        for img_path in sample_test_images:
            print(f"  - {Path(img_path).name}")

else:
    logging.error(f"Model not found at {drive_model_path}. Please ensure the model was saved to Google Drive correctly.")

INFO: Mounting Google Drive to load the model...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
INFO: Loading model from Google Drive: /content/drive/MyDrive/RDD2022_YOLO_Models/best.pt
INFO: Running inference on 5 sample test images...

0: 640x640 4 road_damages, 882.7ms
1: 640x640 1 road_damage, 882.7ms
2: 640x640 (no detections), 882.7ms
3: 640x640 (no detections), 882.7ms
4: 640x640 (no detections), 882.7ms
Speed: 51.8ms preprocess, 882.7ms inference, 32.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/predict_from_drive

Predictions saved to /content/runs/predict_from_drive/ for the following images:
  - China_MotorBike_000376.jpg
  - Norway_008147.jpg
  - Norway_006514.jpg
  - United_States_003096.jpg
  - India_003083.jpg


In [21]:
from ultralytics import YOLO
from pathlib import Path
import logging
from google.colab import drive

# Re-define IMG_SIZE and BATCH_SIZE from cell 10ba6e98
IMG_SIZE = 640
BATCH_SIZE = 16

logging.info("Mounting Google Drive to load the model for accuracy evaluation...")
drive.mount('/content/drive')

# Define the path to the model saved on Google Drive
drive_model_path = Path('/content/drive/MyDrive/RDD2022_YOLO_Models/best.pt')

if drive_model_path.exists():
    logging.info(f"Loading model from Google Drive: {drive_model_path}")
    loaded_model_for_accuracy = YOLO(drive_model_path)

    # Run validation on the test set
    # The DATASET_PATH and data.yaml should already be set up from previous steps
    logging.info("Calculating accuracy on the test set...")
    metrics = loaded_model_for_accuracy.val(
        data=str(DATASET_PATH / "data.yaml"),
        split="test",
        imgsz=IMG_SIZE, # Use the same image size as during training
        batch=BATCH_SIZE # Use the same batch size as during training
    )

    print("\n--- Model Accuracy Metrics on Test Set ---")
    # Access scalar values using .item() if they are numpy arrays
    print(f"mAP50:    {metrics.box.map50.item():.4f}")
    print(f"mAP50-95: {metrics.box.map.item():.4f}")
    print(f"Precision: {metrics.box.p.item():.4f}")
    print(f"Recall: {metrics.box.r.item():.4f}")

else:
    logging.error(f"Model not found at {drive_model_path}. Cannot evaluate accuracy.")

INFO: Mounting Google Drive to load the model for accuracy evaluation...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
INFO: Loading model from Google Drive: /content/drive/MyDrive/RDD2022_YOLO_Models/best.pt
INFO: Calculating accuracy on the test set...
Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1528.2±528.9 MB/s, size: 210.0 KB)
val: Scanning /content/dataset/test/labels.cache... 937 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 937/937 187.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 59/59 4.6s/it 4:30
                   all        937       1834      0.707        0.6      0.641      0.367
Speed: 6.7ms preprocess, 264.5ms inference, 0.0ms loss, 0.6ms pos